<a href="https://colab.research.google.com/github/yejinPARK48/michigan_building_detection/blob/main/06_Puma_estimation/Phase6_2601600.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big PUMA 2601600 — Building Estimation (parallel)

Single PUMA, **~102,888 tiles**. NAIP downloads run in parallel across image groups
(`workers` threads); inference runs batched on the GPU after each chunk.
Checkpointed per image-group for disconnect resilience.

## 0. Setup

In [ ]:
# SSL patch MUST be first, before any other network imports
import os, ssl, urllib3
os.environ['CURL_CA_BUNDLE']     = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()


In [ ]:
!pip install pystac-client planetary-computer rasterio geopandas shapely pyogrio -q
!pip install segmentation-models-pytorch albumentations opencv-python-headless scikit-image -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.2 MB/s eta 0:00:00


In [ ]:
import os, warnings, zipfile, urllib.request, time, shutil, json, logging
warnings.filterwarnings('ignore')
os.environ['GDAL_HTTP_MAX_RETRY']   = '5'
os.environ['GDAL_HTTP_RETRY_DELAY'] = '1'
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from PIL import Image
from shapely.geometry import box, Point, shape as shapely_shape
from shapely.ops import transform as shp_transform
from shapely.prepared import prep
from skimage import measure

import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.crs import CRS
from rasterio.warp import transform_bounds
logging.getLogger('rasterio._err').setLevel(logging.ERROR)
logging.getLogger('rasterio').setLevel(logging.ERROR)

import pystac_client
import planetary_computer as pc
pc.settings.set_subscription_key("0162e7782ab28b85cda3c77b61875ca3210a1c31")

import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from google.colab import drive
drive.mount('/content/drive')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Mounted at /content/drive
Device: cuda


## 1. Config

In [ ]:
PUMAS_9 = [
    '2601200', '2603203', '2600100', '2602903', '2600802',
    '2603212', '2601701', '2601703', '2601600',
]

BASE_DIR      = '/content/drive/MyDrive/michigan_unet_project'
CKPT_DIR      = f'{BASE_DIR}/checkpoints'
SEG_CKPT      = f'{CKPT_DIR}/unet_seg_best_full_curated.pt'

PRED_DIR      = f'{BASE_DIR}/results_9puma/predictions_puma'
PROGRESS_DIR  = f'{BASE_DIR}/results_9puma/progress_puma'
GEO_DIR       = f'{BASE_DIR}/geo'

for d in [PRED_DIR, PROGRESS_DIR, GEO_DIR]:
    os.makedirs(d, exist_ok=True)

TILE_SIZE_M    = 256
TILE_SIZE_PX   = 512
UTM_EPSG       = 26917
NAIP_YEAR      = 2020
BEST_THRESHOLD = 0.5
BEST_MIN_AREA  = 50
BATCH_SIZE     = 32   # inference batch size (raise to 48-64 on L4/A100)

TILE_AREA_KM2 = (TILE_SIZE_M ** 2) / 1e6          # ground area per tile
PIXEL_AREA_M2 = (TILE_SIZE_M / TILE_SIZE_PX) ** 2  # ground area per pixel

DENSE_THRESHOLD  = 0.05
SPARSE_THRESHOLD = 0.001
def classify_tier(ratio):
    if ratio >= DENSE_THRESHOLD:    return 'Dense'
    elif ratio >= SPARSE_THRESHOLD: return 'Sparse'
    else:                           return 'Empty'

print('Checkpoint exists:', os.path.exists(SEG_CKPT))


Checkpoint exists: True


## 2. Load PUMA boundaries + ranking (once)

In [ ]:
def fetch_tiger(candidate_urls, local_zip, extract_dir):
    # download the first reachable TIGER zip, extract, return dir
    if any(f.endswith('.shp') for f in os.listdir(extract_dir)):
        return extract_dir
    last = None
    for url in candidate_urls:
        try:
            print('Trying', url)
            urllib.request.urlretrieve(url, local_zip)
            with zipfile.ZipFile(local_zip) as z:
                z.extractall(extract_dir)
            return extract_dir
        except Exception as e:
            last = e
            print('  failed:', e)
    raise RuntimeError(f'all PUMA URLs failed: {last}')

PUMA_DIR = f'{GEO_DIR}/puma20'
os.makedirs(PUMA_DIR, exist_ok=True)
fetch_tiger(
    candidate_urls=[
        'https://www2.census.gov/geo/tiger/TIGER2023/PUMA20/tl_2023_26_puma20.zip',
        'https://www2.census.gov/geo/tiger/TIGER2022/PUMA20/tl_2022_26_puma20.zip',
        'https://www2.census.gov/geo/tiger/TIGER2024/PUMA20/tl_2024_26_puma20.zip',
    ],
    local_zip=f'{PUMA_DIR}/puma20.zip',
    extract_dir=PUMA_DIR,
)
puma_shp = [f'{PUMA_DIR}/{f}' for f in os.listdir(PUMA_DIR) if f.endswith('.shp')][0]
gdf_puma_all = gpd.read_file(puma_shp)
geoid_col = next(c for c in gdf_puma_all.columns if c.upper().startswith('GEOID'))
aland_col = next((c for c in gdf_puma_all.columns if c.upper().startswith('ALAND')), None)
gdf_puma_all[geoid_col] = gdf_puma_all[geoid_col].astype(str)

_r = gdf_puma_all[gdf_puma_all[geoid_col].isin(PUMAS_9)].copy()
if aland_col:
    _r['area_km2'] = _r[aland_col] / 1e6
else:
    _r['area_km2'] = _r.to_crs(epsg=UTM_EPSG).geometry.area / 1e6
_r = _r[[geoid_col, 'area_km2']].sort_values('area_km2').reset_index(drop=True)
_r['est_tiles'] = (_r['area_km2'] * 1e6 / (TILE_SIZE_M ** 2)).round().astype(int)
print('9 PUMAs ranked smallest -> largest:')
print(_r.to_string(index=False))


9 PUMAs ranked smallest -> largest:
GEOID20     area_km2  est_tiles
2603212    65.106587        993
2603203    93.779840       1431
2601703   163.192431       2490
2602903   177.935191       2715
2600802   846.509173      12917
2601701  2131.045270      32517
2601200  4417.097111      67400
2601600  6742.852277     102888
2600100 22265.787264     339749


## 3. Load statewide block groups (once)

In [ ]:
SHP_BG = f'{BASE_DIR}/tl_2020_26_bg.shp'
if not os.path.exists(SHP_BG):
    print('Block-group shapefile not in Drive - downloading...')
    urllib.request.urlretrieve(
        'https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_26_bg.zip',
        '/content/tl_2020_26_bg.zip')
    with zipfile.ZipFile('/content/tl_2020_26_bg.zip') as z:
        z.extractall(BASE_DIR)

gdf_bg = gpd.read_file(SHP_BG).to_crs(epsg=4326)
gdf_bg = gdf_bg.rename(columns={'GEOID': 'bg_geoid'})
gdf_bg['bg_geoid'] = gdf_bg['bg_geoid'].astype(str)
print('Statewide block groups loaded:', len(gdf_bg))


Statewide block groups loaded: 8386


## 4. Load model (once)

In [ ]:
model = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                 in_channels=3, classes=1, activation=None).to(DEVICE)
model.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE))
model.eval()
torch.backends.cudnn.benchmark = True   # fixed 512x512 input -> faster convolutions
print('Model loaded.')

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class InferenceDataset(Dataset):
    def __init__(self, tile_ids, img_dir, transform=None):
        self.tile_ids, self.img_dir, self.transform = tile_ids, img_dir, transform
    def __len__(self): return len(self.tile_ids)
    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        img = np.array(Image.open(f'{self.img_dir}/{tid}.png').convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, tid

def count_and_area(pred_mask):
    # building count + footprint area (m^2) for components >= BEST_MIN_AREA px
    # (np.bincount over the labeled array is much faster than regionprops, same result)
    labeled = measure.label(pred_mask)
    if labeled.max() == 0:
        return 0, 0.0
    sizes = np.bincount(labeled.ravel())[1:]        # pixel count per component
    big = sizes[sizes >= BEST_MIN_AREA]
    return int(big.size), float(big.sum()) * PIXEL_AREA_M2


Model loaded.


## 5. Helper functions + aggregation

In [ ]:
def get_naip_items(bbox, year=NAIP_YEAR, max_items=4000):
    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1', modifier=pc.sign_inplace)
    items = catalog.search(collections=['naip'], bbox=bbox,
                           datetime=f'{year}-01-01/{year}-12-31',
                           max_items=max_items).item_collection()
    if len(items) == 0:
        items = catalog.search(collections=['naip'], bbox=bbox,
                               max_items=max_items).item_collection()
    return list(items)

def build_item_index(items):
    recs = []
    for it in items:
        try:
            recs.append({'item_id': it.id,
                         'href': it.assets['image'].href,
                         'geometry': shapely_shape(it.geometry)})
        except Exception:
            continue
    return gpd.GeoDataFrame(recs, geometry='geometry', crs='EPSG:4326')

def crop_tile_from_src(src, bbox, size_px=TILE_SIZE_PX):
    try:
        bt = transform_bounds(CRS.from_epsg(4326), src.crs, *bbox)
        window = from_bounds(*bt, transform=src.transform)
        data = src.read([1, 2, 3], window=window, out_shape=(3, size_px, size_px),
                        resampling=Resampling.bilinear)
        return np.transpose(data, (1, 2, 0)).astype(np.uint8)
    except Exception:
        return None

def fetch_naip_tile(bbox, year=NAIP_YEAR, size_px=TILE_SIZE_PX, max_retries=3):
    for attempt in range(max_retries):
        try:
            catalog = pystac_client.Client.open(
                'https://planetarycomputer.microsoft.com/api/stac/v1', modifier=pc.sign_inplace)
            items = catalog.search(collections=['naip'], bbox=bbox,
                                   datetime=f'{year}-01-01/{year}-12-31',
                                   max_items=5).item_collection()
            if len(items) == 0:
                items = catalog.search(collections=['naip'], bbox=bbox,
                                       max_items=5).item_collection()
                if len(items) == 0:
                    return None
            href = pc.sign(items[0].assets['image'].href)
            with rasterio.open(href) as src:
                return crop_tile_from_src(src, bbox, size_px)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return None
    return None

def build_puma_tiles(puma_code, puma_geom, gdf_bg_puma,
                     tile_size_m=TILE_SIZE_M, crs_utm=UTM_EPSG):
    # one regular grid over the PUMA polygon; keep tiles whose centroid is inside
    to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{crs_utm}', always_xy=True).transform
    to_wgs = Transformer.from_crs(f'EPSG:{crs_utm}', 'EPSG:4326', always_xy=True).transform
    geom_utm = shp_transform(to_utm, puma_geom)
    pgeom = prep(geom_utm)
    minx, miny, maxx, maxy = geom_utm.bounds

    rows, seq, x = [], 0, minx
    while x < maxx:
        y = miny
        while y < maxy:
            cx, cy = x + tile_size_m / 2, y + tile_size_m / 2
            if pgeom.contains(Point(cx, cy)):
                wx0, wy0 = to_wgs(x, y)
                wx1, wy1 = to_wgs(x + tile_size_m, y + tile_size_m)
                wcx, wcy = to_wgs(cx, cy)
                rows.append((f'{puma_code}_{seq:06d}',
                             min(wx0, wx1), min(wy0, wy1), max(wx0, wx1), max(wy0, wy1),
                             wcx, wcy))
                seq += 1
            y += tile_size_m
        x += tile_size_m

    df = pd.DataFrame(rows, columns=['tile_id', 'minx', 'miny', 'maxx', 'maxy', 'cx', 'cy'])
    gt = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['cx'], df['cy']), crs='EPSG:4326')
    j = gpd.sjoin(gt[['tile_id', 'geometry']],
                  gdf_bg_puma[['bg_geoid', 'county_fips', 'geometry']],
                  how='left', predicate='within')
    j = j[~j.index.duplicated(keep='first')]
    gt['bg_geoid']    = j['bg_geoid'].values
    gt['county_fips'] = j['county_fips'].values
    gt = gt.dropna(subset=['bg_geoid']).reset_index(drop=True)
    gt['bbox'] = list(zip(gt['minx'], gt['miny'], gt['maxx'], gt['maxy']))
    return gt

def aggregate_bg(puma_code, gdf_bg_puma, pred_path, bg_path):
    if not os.path.exists(pred_path):
        print(f'[{puma_code}] no predictions to aggregate')
        return
    # use official land area (ALAND) when available so density excludes open water;
    # fall back to projected polygon area otherwise
    _ac = next((cc for cc in gdf_bg_puma.columns if cc.upper().startswith('ALAND')), None)
    if _ac:
        _land = gdf_bg_puma[_ac].astype(float).values / 1e6
    else:
        _land = gdf_bg_puma.to_crs(epsg=UTM_EPSG).geometry.area.values / 1e6
    bg_area = pd.DataFrame({'bg_geoid': gdf_bg_puma['bg_geoid'].values,
                            'county_fips': gdf_bg_puma['county_fips'].values,
                            'area_km2': _land})
    df = pd.read_csv(pred_path, dtype={'bg_geoid': str, 'county_fips': str})
    bg = (df.groupby('bg_geoid')
            .agg(n_tiles=('tile_id', 'count'),
                 pred_buildings=('pred_count', 'sum'),
                 mask_area_m2=('mask_area_m2', 'sum'),
                 mean_mask_ratio=('mask_ratio', 'mean'),
                 dense_tiles=('tier', lambda s: (s == 'Dense').sum()),
                 sparse_tiles=('tier', lambda s: (s == 'Sparse').sum()),
                 empty_tiles=('tier', lambda s: (s == 'Empty').sum()))
            .reset_index())
    bg = bg.merge(bg_area, on='bg_geoid', how='left')
    bg['puma'] = puma_code
    bg['mask_area_km2']        = bg['mask_area_m2'] / 1e6
    bg['tiled_area_km2']       = bg['n_tiles'] * TILE_AREA_KM2
    bg['built_fraction']       = bg['mask_area_km2'] / bg['area_km2']
    bg['built_pct']            = bg['built_fraction'] * 100
    bg['building_density_km2'] = bg['pred_buildings'] / bg['area_km2']
    for col, nd in [('area_km2', 4), ('tiled_area_km2', 4), ('mask_area_km2', 5),
                    ('built_fraction', 5), ('built_pct', 2),
                    ('building_density_km2', 2), ('mean_mask_ratio', 6)]:
        bg[col] = bg[col].round(nd)
    bg = bg[['puma', 'bg_geoid', 'county_fips', 'area_km2', 'tiled_area_km2',
             'pred_buildings', 'building_density_km2', 'mask_area_km2',
             'built_fraction', 'built_pct', 'n_tiles', 'mean_mask_ratio',
             'dense_tiles', 'sparse_tiles', 'empty_tiles']]
    bg.to_csv(bg_path, index=False)
    tb, ta, tm = bg['pred_buildings'].sum(), bg['area_km2'].sum(), bg['mask_area_km2'].sum()
    print(f'[{puma_code}] -> {bg_path}')
    print(f'[{puma_code}] buildings {tb:,} | land {ta:.1f} km^2 | '
          f'density {tb/ta:.1f}/km^2 | built-up {100*tm/ta:.2f}%')


## 6. process_puma() — full per-PUMA pipeline

In [ ]:
def process_puma(puma_code, parallel=False, workers=8):
    pred_path     = f'{PRED_DIR}/tile_predictions_puma_{puma_code}.csv'
    bg_path       = f'{PRED_DIR}/bg_estimates_puma_{puma_code}.csv'
    progress_path = f'{PROGRESS_DIR}/done_images_{puma_code}.json'
    img_dir       = f'/content/tiles_tmp/{puma_code}'

    if os.path.exists(bg_path):
        print(f'[{puma_code}] already complete; skipping')
        return

    sel = gdf_puma_all[gdf_puma_all[geoid_col] == puma_code]
    if len(sel) == 0:
        print(f'[{puma_code}] not found in PUMA shapefile; skipping')
        return
    puma_geom = sel.to_crs(epsg=4326).iloc[0].geometry

    bg_c = gdf_bg.copy()
    bg_c['geometry'] = gdf_bg.geometry.centroid
    gdf_bg_puma = gdf_bg[bg_c.geometry.within(puma_geom)].reset_index(drop=True)
    gdf_bg_puma['county_fips'] = '26' + gdf_bg_puma['bg_geoid'].str[2:5]

    gdf_tiles = build_puma_tiles(puma_code, puma_geom, gdf_bg_puma)
    print(f'[{puma_code}] block groups {len(gdf_bg_puma)} | tiles {len(gdf_tiles):,}')
    if len(gdf_tiles) == 0:
        return

    naip_items = get_naip_items(puma_geom.bounds)
    gdf_items  = build_item_index(naip_items)
    cent = gdf_tiles[['tile_id', 'geometry']].copy()
    ti = gpd.sjoin(cent, gdf_items[['item_id', 'geometry']], how='left', predicate='within')
    ti = ti[~ti.index.duplicated(keep='first')]
    gdf_tiles['item_id'] = ti['item_id'].values
    n_assigned = gdf_tiles['item_id'].notna().sum()
    print(f'[{puma_code}] NAIP images {len(gdf_items)} | assigned {n_assigned:,} | '
          f'fallback {len(gdf_tiles) - n_assigned:,}')

    # NAIP only covers land. A large uncovered fraction means open water
    # (e.g., the Great Lakes) with no imagery and no buildings -- drop those tiles
    # instead of running a slow, pointless per-tile search over the lake.
    uncovered = int(gdf_tiles['item_id'].isna().sum())
    if len(gdf_tiles) and uncovered / len(gdf_tiles) > 0.02:
        print(f'[{puma_code}] dropping {uncovered:,} water/no-NAIP tiles; '
              f'keeping {len(gdf_tiles) - uncovered:,} land tiles')
        gdf_tiles = gdf_tiles[gdf_tiles['item_id'].notna()].reset_index(drop=True)

    done_images = set(json.load(open(progress_path))) if os.path.exists(progress_path) else set()
    done_tiles  = set(pd.read_csv(pred_path, usecols=['tile_id'])['tile_id']) if os.path.exists(pred_path) else set()
    tile_meta = gdf_tiles.set_index('tile_id')[['bg_geoid', 'county_fips']]

    def append_predictions(rows):
        if not rows:
            return
        d = pd.DataFrame(rows).join(tile_meta, on='tile_id')
        d['puma'] = puma_code
        d = d[['tile_id', 'puma', 'bg_geoid', 'county_fips',
               'pred_count', 'mask_area_m2', 'mask_ratio', 'tier']]
        d.to_csv(pred_path, mode='a', header=not os.path.exists(pred_path), index=False)

    def predict_saved(tids):
        if not tids:
            return []
        ld = DataLoader(InferenceDataset(tids, img_dir, val_aug),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
                        pin_memory=(DEVICE == 'cuda'))
        out = []
        with torch.no_grad():
            for imgs, t in ld:
                imgs = imgs.to(DEVICE, non_blocking=True)
                with torch.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
                    logits = model(imgs)
                probs  = torch.sigmoid(logits.float())
                masks  = (probs > BEST_THRESHOLD).cpu().numpy()
                ratios = masks.reshape(masks.shape[0], -1).mean(axis=1)
                for b in range(len(t)):
                    cnt, area = count_and_area(masks[b, 0])
                    out.append({'tile_id': t[b], 'pred_count': cnt,
                                'mask_area_m2': round(area, 2),
                                'mask_ratio': round(float(ratios[b]), 6),
                                'tier': classify_tier(float(ratios[b]))})
        return out

    def download_group(item_id, grp):
        href = None
        if item_id is not None and not (isinstance(item_id, float) and np.isnan(item_id)):
            hh = gdf_items.loc[gdf_items['item_id'] == item_id, 'href']
            href = hh.iloc[0] if len(hh) else None

        pending = [(r['tile_id'], r['bbox']) for _, r in grp.iterrows()
                   if r['tile_id'] not in done_tiles]
        saved = []

        # try the shared image up to 3x, RE-SIGNING the SAS token each attempt.
        # a 403 partway through usually means the token expired or got throttled,
        # so a fresh pc.sign + short backoff recovers it.
        for attempt in range(3):
            if not pending or href is None:
                break
            src = None
            try:
                src = rasterio.open(pc.sign(href))
            except Exception:
                src = None
            still = []
            for tid, bbox in pending:
                img = crop_tile_from_src(src, bbox) if src is not None else None
                if img is None or img.sum() == 0:
                    still.append((tid, bbox))
                    continue
                Image.fromarray(img).save(f'{img_dir}/{tid}.png')
                saved.append(tid)
            if src is not None:
                src.close()
            pending = still
            if pending:
                time.sleep(2 * (attempt + 1))

        # anything still missing: per-tile STAC search (fresh sign per tile)
        for tid, bbox in pending:
            img = fetch_naip_tile(bbox)
            if img is not None:
                Image.fromarray(img).save(f'{img_dir}/{tid}.png')
                saved.append(tid)
        return saved

    def flush(saved, finished):
        append_predictions(predict_saved(saved))
        done_tiles.update(saved)
        done_images.update(finished)
        json.dump(sorted(str(x) for x in done_images), open(progress_path, 'w'))

    groups = [(iid, g) for iid, g in gdf_tiles.groupby('item_id', dropna=False)]
    total = len(groups)
    shutil.rmtree(img_dir, ignore_errors=True); os.makedirs(img_dir, exist_ok=True)

    if parallel:
        pending = [(iid, g) for iid, g in groups if str(iid) not in done_images]
        for s in range(0, len(pending), workers):
            chunk = pending[s:s + workers]
            saved_all = []
            with ThreadPoolExecutor(max_workers=workers) as ex:
                for fut in [ex.submit(download_group, iid, g) for iid, g in chunk]:
                    saved_all.extend(fut.result())
            flush(saved_all, [str(iid) for iid, _ in chunk])
            shutil.rmtree(img_dir, ignore_errors=True); os.makedirs(img_dir, exist_ok=True)
            print(f'[{puma_code}] {min(s + workers, len(pending))}/{len(pending)} '
                  f'image-groups | +{len(saved_all):,} tiles')
    else:
        for gi, (iid, g) in enumerate(groups, 1):
            if str(iid) in done_images:
                continue
            saved = download_group(iid, g)
            flush(saved, [str(iid)])
            shutil.rmtree(img_dir, ignore_errors=True); os.makedirs(img_dir, exist_ok=True)
            print(f'[{puma_code}] [{gi}/{total}] image {iid} | +{len(saved):,} tiles')

    shutil.rmtree(img_dir, ignore_errors=True)
    aggregate_bg(puma_code, gdf_bg_puma, pred_path, bg_path)
    print(f'[{puma_code}] DONE')


## 7. Run — PUMA 2601600 (parallel)

In [ ]:
PUMA_CODE = '2601600'   # ~102,888 tiles
WORKERS   = 10        # parallel NAIP downloads

process_puma(PUMA_CODE, parallel=True, workers=WORKERS)


[2601600] block groups 128 | tiles 183,334
[2601600] NAIP images 267 | assigned 116,954 | fallback 66,380
[2601600] dropping 66,380 water/no-NAIP tiles; keeping 116,954 land tiles
[2601600] 10/97 image-groups | +5,484 tiles
[2601600] 20/97 image-groups | +4,414 tiles
[2601600] 30/97 image-groups | +5,450 tiles
[2601600] 40/97 image-groups | +4,449 tiles
[2601600] 50/97 image-groups | +5,349 tiles
[2601600] 60/97 image-groups | +5,022 tiles
[2601600] 70/97 image-groups | +4,835 tiles
[2601600] 80/97 image-groups | +3,709 tiles
[2601600] 90/97 image-groups | +2,655 tiles
[2601600] 97/97 image-groups | +3,423 tiles
[2601600] -> /content/drive/MyDrive/michigan_unet_project/results_9puma/predictions_puma/bg_estimates_puma_2601600.csv
[2601600] buildings 118,850 | land 6742.9 km^2 | density 17.6/km^2 | built-up 0.42%
[2601600] DONE
